# Conversational Summarizer
- #### Multi-turn conversation with memory
- #### Context-aware responses

I'll build this as a runnable Python script using LangChain's current (non-deprecated) patterns: `RunnableWithMessageHistory` for multi-turn memory, plus a rolling summarization step so the bot stays context-aware without the prompt growing unbounded.How it works:

- **Memory**: `InMemoryChatMessageHistory` per `session_id`, wired in via `RunnableWithMessageHistory` — this is the modern replacement for the deprecated `ConversationBufferMemory`.
- **Context-aware responses**: each call injects both the running `summary` and the recent raw message history into the prompt, so the model sees long-term context plus the exact recent wording.
- **Summarizer**: once raw history passes a threshold (8 messages here), it's condensed into a rolling summary via a second small chain, and the raw buffer is trimmed to the last few turns — so the context never grows unbounded even in very long conversations.

Two things worth tuning for real use:
1. `token_counter=len` in `trim_messages` counts *messages*, not tokens — swap in a `tiktoken`-based counter for accurate context-window management.
2. `InMemoryChatMessageHistory`/`_summaries` are process-local dicts — for a real app, back these with Redis, Postgres, or another persistent store (LangChain has built-in integrations for several) so history survives restarts and works across multiple server instances.

## Using Langchain

In [2]:
"""
Conversational Summarizer — LangChain + OpenAI
------------------------------------------------
Features:
  1. Multi-turn conversation memory (per session_id).
  2. A rolling summary of the conversation, updated every N turns,
     so responses stay context-aware even after the raw history is trimmed.
  3. Context-aware responses: each reply is generated using both the
     running summary AND the most recent raw turns.

Install:
    pip install langchain langchain-openai

Set your key:
    export OPENAI_API_KEY="sk-..."
"""

import os
from typing import Dict

from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.messages import trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

# ---------------------------------------------------------------------------
# 1. Model
# ---------------------------------------------------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)  # swap for any chat model you have access to

# ---------------------------------------------------------------------------
# 2. Per-session raw message history (the "memory")
# ---------------------------------------------------------------------------
_store: Dict[str, BaseChatMessageHistory] = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in _store:
        _store[session_id] = InMemoryChatMessageHistory()
    return _store[session_id]


# ---------------------------------------------------------------------------
# 3. Rolling summary — keeps the conversation "context-aware" long-term
# ---------------------------------------------------------------------------
_summaries: Dict[str, str] = {}

_summary_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You maintain a running summary of an ongoing conversation. "
            "Given the previous summary and the new turns, write an updated, "
            "concise summary (under 150 words) capturing key facts, decisions, "
            "and user preferences. Do not lose important details from the previous summary.",
        ),
        (
            "human",
            "Previous summary:\n{previous_summary}\n\nNew turns:\n{new_turns}\n\nUpdated summary:",
        ),
    ]
)

_summarizer_chain = _summary_prompt | llm | StrOutputParser()


def update_summary(session_id: str, new_turns_text: str) -> str:
    previous = _summaries.get(session_id, "No previous summary.")
    updated = _summarizer_chain.invoke({"previous_summary": previous, "new_turns": new_turns_text})
    _summaries[session_id] = updated
    return updated


# ---------------------------------------------------------------------------
# 4. Main chat chain — uses summary + recent raw history for each reply
# ---------------------------------------------------------------------------
_chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful, context-aware assistant. Use the running summary "
            "below together with the recent message history to stay consistent "
            "with what the user has already told you.\n\n"
            "Conversation summary so far:\n{summary}",
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

_chat_chain = _chat_prompt | llm | StrOutputParser()

conversational_chain = RunnableWithMessageHistory(
    _chat_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# Trigger a summary refresh + history trim once raw history passes this many messages.
# In production, count actual tokens (e.g. with tiktoken) instead of message count.
HISTORY_MESSAGE_LIMIT = 8
MESSAGES_TO_KEEP_AFTER_TRIM = 4


def chat(session_id: str, user_input: str) -> str:
    """Send one user turn and get a context-aware reply."""
    summary = _summaries.get(session_id, "No conversation yet.")

    response = conversational_chain.invoke(
        {"input": user_input, "summary": summary},
        config={"configurable": {"session_id": session_id}},
    )

    history = get_session_history(session_id)

    if len(history.messages) >= HISTORY_MESSAGE_LIMIT:
        new_turns_text = "\n".join(f"{m.type}: {m.content}" for m in history.messages)
        update_summary(session_id, new_turns_text)

        # Keep only the last few messages verbatim; older detail now lives in the summary.
        trimmed = trim_messages(
            history.messages,
            max_tokens=MESSAGES_TO_KEEP_AFTER_TRIM,
            token_counter=len,   # counts messages, not real tokens — replace for production use
            strategy="last",
        )
        history.clear()
        for m in trimmed:
            history.add_message(m)

    return response


def get_summary(session_id: str) -> str:
    return _summaries.get(session_id, "No summary yet.")


# ---------------------------------------------------------------------------
# 5. Demo
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    session_id = "user-123"
    turns = [
        "Hi, I'm planning a trip to Japan in October.",
        "I love hiking and quiet temples, not crowded tourist traps.",
        "What cities would you recommend?",
        "I also want good ramen recommendations wherever we go.",
        "Given all that, can you draft a rough 5-day itinerary?",
    ]

    for turn in turns:
        print(f"USER: {turn}")
        print(f"BOT : {chat(session_id, turn)}\n")

    print("=== Running Summary ===")
    print(get_summary(session_id))


C:\Users\Mohit\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


USER: Hi, I'm planning a trip to Japan in October.
BOT : That sounds exciting! Japan in October is beautiful, with autumn foliage starting to appear. Do you have specific places in mind that you want to visit, or are you looking for recommendations?

USER: I love hiking and quiet temples, not crowded tourist traps.
BOT : Great! Japan has some wonderful hiking trails and serene temples that are away from the typical tourist crowds. 

For hiking, consider:

1. **Kumano Kodo**: This ancient pilgrimage route in the Kii Peninsula offers stunning scenery and peaceful trails.
2. **Nikko National Park**: Known for its beautiful landscapes and temples, there are plenty of hiking options here, including trails that lead to waterfalls and lakes.
3. **Yakushima**: An island with lush forests and ancient cedar trees, perfect for hiking and enjoying nature.

For quiet temples, you might enjoy:

1. **Tottori Sand Dunes Conan Temple**: A lesser-known temple with beautiful surroundings.
2. **Daitoku-ji

## Withoug using Langchain
Here's the same functionality using only the raw OpenAI SDK — no LangChain, just plain Python data structures for memory and a helper function for summarization.Same design as before, just without LangChain's abstractions:

- **Memory**: a plain dict of lists (`_histories[session_id]`) holding `{"role", "content"}` messages — this replaces `RunnableWithMessageHistory`.
- **Context-aware responses**: each call builds `[system_message_with_summary] + history` and sends it straight to `client.chat.completions.create`.
- **Summarizer**: once history hits `HISTORY_MESSAGE_LIMIT` (8 messages), the older turns are folded into `_summaries[session_id]` via a second API call, and only the last `MESSAGES_TO_KEEP_AFTER_TRIM` messages are kept verbatim — keeping the prompt bounded in long conversations.

Same caveats as before: these dicts are process-local (lost on restart) and the trimming is by message count, not real token count — for production, swap in `tiktoken` to count actual tokens and back the state with a database if you need persistence across restarts or multiple servers.

In [3]:
"""
Conversational Summarizer — plain OpenAI SDK (no LangChain)
-------------------------------------------------------------
Features:
  1. Multi-turn conversation memory (per session_id), stored in-process.
  2. A rolling summary of the conversation, updated every N turns,
     so the model stays context-aware even after older raw turns are dropped.
  3. Context-aware responses: each reply is generated using both the
     running summary AND the most recent raw turns.

Install:
    pip install openai

Set your key:
    export OPENAI_API_KEY="sk-..."
"""

import os
from typing import Dict, List

from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from the environment
MODEL = "gpt-4o-mini"  # swap for any chat model you have access to

# ---------------------------------------------------------------------------
# 1. Per-session state: raw message history + running summary
# ---------------------------------------------------------------------------
# _histories[session_id] = [{"role": "user"/"assistant", "content": "..."}, ...]
_histories: Dict[str, List[dict]] = {}
_summaries: Dict[str, str] = {}

HISTORY_MESSAGE_LIMIT = 8          # trigger a summary refresh once history hits this size
MESSAGES_TO_KEEP_AFTER_TRIM = 4    # keep this many most-recent messages verbatim after trimming


def get_history(session_id: str) -> List[dict]:
    if session_id not in _histories:
        _histories[session_id] = []
    return _histories[session_id]


def get_summary(session_id: str) -> str:
    return _summaries.get(session_id, "No summary yet.")


# ---------------------------------------------------------------------------
# 2. Summarization helper — condenses older turns into a running summary
# ---------------------------------------------------------------------------
def update_summary(session_id: str, turns_to_summarize: List[dict]) -> str:
    previous_summary = _summaries.get(session_id, "No previous summary.")
    turns_text = "\n".join(f"{m['role']}: {m['content']}" for m in turns_to_summarize)

    messages = [
        {
            "role": "system",
            "content": (
                "You maintain a running summary of an ongoing conversation. "
                "Given the previous summary and the new turns, write an updated, "
                "concise summary (under 150 words) capturing key facts, decisions, "
                "and user preferences. Do not lose important details from the previous summary."
            ),
        },
        {
            "role": "user",
            "content": f"Previous summary:\n{previous_summary}\n\nNew turns:\n{turns_text}\n\nUpdated summary:",
        },
    ]

    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.3)
    updated = response.choices[0].message.content.strip()
    _summaries[session_id] = updated
    return updated


# ---------------------------------------------------------------------------
# 3. Main chat function — context-aware via summary + recent raw history
# ---------------------------------------------------------------------------
def chat(session_id: str, user_input: str) -> str:
    history = get_history(session_id)
    summary = get_summary(session_id)

    system_message = {
        "role": "system",
        "content": (
            "You are a helpful, context-aware assistant. Use the running summary "
            "below together with the recent message history to stay consistent "
            "with what the user has already told you.\n\n"
            f"Conversation summary so far:\n{summary}"
        ),
    }

    history.append({"role": "user", "content": user_input})

    messages = [system_message] + history
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.3)
    reply = response.choices[0].message.content.strip()

    history.append({"role": "assistant", "content": reply})

    # Once raw history grows past the limit, fold older turns into the summary
    # and trim the buffer so the prompt doesn't grow unbounded.
    if len(history) >= HISTORY_MESSAGE_LIMIT:
        turns_to_summarize = history[:-MESSAGES_TO_KEEP_AFTER_TRIM]
        update_summary(session_id, turns_to_summarize)
        _histories[session_id] = history[-MESSAGES_TO_KEEP_AFTER_TRIM:]

    return reply


# ---------------------------------------------------------------------------
# 4. Demo
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    session_id = "user-123"
    turns = [
        "Hi, I'm planning a trip to Japan in October.",
        "I love hiking and quiet temples, not crowded tourist traps.",
        "What cities would you recommend?",
        "I also want good ramen recommendations wherever we go.",
        "Given all that, can you draft a rough 5-day itinerary?",
    ]

    for turn in turns:
        print(f"USER: {turn}")
        print(f"BOT : {chat(session_id, turn)}\n")

    print("=== Running Summary ===")
    print(get_summary(session_id))

USER: Hi, I'm planning a trip to Japan in October.
BOT : That sounds exciting! Do you have specific places in Japan that you want to visit, or are you looking for recommendations?

USER: I love hiking and quiet temples, not crowded tourist traps.
BOT : Great! Japan has many beautiful hiking trails and serene temples that are away from the typical tourist crowds. Some recommendations include:

1. **Kumano Kodo** - A network of ancient pilgrimage routes in the Kii Peninsula, offering stunning scenery and peaceful temples along the way.

2. **Nikko** - While some areas can be busy, there are quieter trails and beautiful temples like Toshogu Shrine set in a lush forest.

3. **Mount Takao** - Just outside of Tokyo, it offers several hiking trails and a lovely temple at the summit, with fewer crowds than more famous spots.

4. **Shikoku Pilgrimage** - A long-distance trail that connects 88 temples around Shikoku Island, allowing for a mix of hiking and spiritual exploration.

5. **Koya-san**